# Polar chip creation and edge-case validation

This is the polar-focused counterpart to [chip_example.ipynb](chip_example.ipynb), using the same chip backend. Small north/south AOIs may cross ±82° or the antimeridian. Full-longitude pole caps are supported on polar-stereographic grids: the enclosing rectangle retains imagery and labels in its corners, without a circular mask. Pole integration tests and real-data review require HPC validation.

The user reports the updated chip tests passed on HPC, including seam regressions. The north seam has user-reported visual acceptance; pole and other real-data combinations remain pending. Edit source/label paths and AOIs together; latest-label discovery does not verify spatial association.

Dynamic output retains the original raster CRS and pixel lattice. Static-only output uses a 100 m, zero-anchored lattice in the AOI-center grid (LPS for polar centers). Acquisition tiles are intermediate, not the output grid. Select the project's GDAL-enabled kernel.

Steps 1–4 preserve the working example. Optional cases below extend the same batch; step 5 checks real-data serial/parallel reproducibility, and step 6 runs synthetic regressions in a separate Python process. These are complementary checks, not proof of every possible combination. No supplied labels are assumed complete at the antimeridian, south pole, or immediate north pole.


## Imports and repository discovery

In [ ]:
from pathlib import Path
from datetime import datetime
import sys

repo_root = Path(str(Path.cwd().parent).replace(
    "/panfs/ccds02/nobackup", "/explore/nobackup"))
if not (repo_root / "lfm/data_processing").is_dir():
    raise FileNotFoundError("Run from the repository's top-level notebooks/ directory.")
sys.path.insert(0, str(repo_root))

from lfm.data_processing.chip import (
    AcquisitionGroupConfig, ChipConfig, GeographicAOI, LabelInput,
    NoSplitConfig, OutputModalityConfig, SourceSelector,
    chip_grid_family, default_chip_zoom, chip_request_from_aoi, create_chips,
)
from lfm.data_processing.tiling import (
    TileConfig, TileSourceConfig, VectorIndexBuildConfig,
    ensure_vector_index, resolve_notebook_source_index, discover_raster_paths,
)
from lfm.data_processing.tiling.product_ids import lunar_product_id_from_raster_path
from lfm.data_processing.tiling import make_static_source
from lfm.data_processing.chip.chip_notebook_utils import (
    latest_crater_label_path, read_source_grid, plot_chip_result, chip_batch_fingerprints,
)


## 1. Configure one study area

Settings follow the same groups and order as the code below. Start with study setup, imagery and labels; enable additional AOIs only after confirming coverage.

### Study setup

- `PROJECT_DATA_DIR`: Root of the shared LFM project data.
- `HEMISPHERE`: `north` or `south`; selects example bounds and the WAC discovery region, but does not change the label path.
- `MODALITY`: `wac`, `nac`, or `static` (static-only ignores dynamic imagery). **Note: there doesn't seem to be a polar NAC .tif file in the LFM project space at the moment. Thus, nac hasn't been tested for the polar workflow, but feel free to give it a try.**

### Imagery inputs

- `INCLUDE_STATIC`: Append the canonical 63 static bands to dynamic imagery; ignored for static-only runs. 
- `SOURCE_RASTER`: Original pre-tiling TIFF/VRT. `None` defaults to the WAC .vrt file found in the static data subdirectory of `processed_data`. NAC and VIS/UV WAC require an explicit path.
- `WAC_LAYOUT`: `single` for the monochrome polar mosaic, or `vis_uv` for five VIS plus two UV bands. VIS/UV needs a five-band VIS reference and its UV companion.
- `DYNAMIC_GLOB`: Optional companion-file pattern; defaults to the selected raster or the product VIS/UV pattern. Never index both a VRT and its backing TIFF.
- `POLAR_WAC_DIR`: Collection searched for automatic polar WAC VRT discovery.
- `STATIC_DATA_DIR`: Canonical static-raster collection, used when static imagery is enabled.

### Crater labels

- `LABEL_PATH`: Finished GPKG/TIFF, or `None` for the latest crater-notebook GPKG. Discovery does not verify hemisphere or product; the default file is for the north example.
- `LABEL_KIND`: `auto` detects GPKGs and treats TIFFs as semantic. Use `raster_instance` for instance-ID TIFFs, or `semantic` explicitly.
- `LABEL_LAYER`: Polygon layer within a GPKG; ignored for TIFFs.

### Study area and additional AOIs

- `AOI_NWSE`: Main IAU:30100 bounds in **north, west, south, east** order. The north default encloses craters 13–20 in the supplied polar labels; south needs its own labels. A commented antimeridian alternative is provided.
- `RUN_SEAM_EXAMPLE`: Add a seam-crossing chip to the same batch; enabled for north WAC by default.
- `SEAM_AOI_NWSE`: Additional bounds extending below 82°N. Adjust or disable for other imagery/labels.
- `RUN_POLE_EXAMPLE`: Add a pole-containing chip; disabled until imagery and label coverage is confirmed.
- `POLE_AOI_NWSE`: Full-longitude cap: west=-180, east=180, reaching ±90°. Labels must cover the enclosing output rectangle, including corners outside the cap.

### Execution and outputs

- `SAMPLE_ID`: Unique name for the main study-area chip.
- `ZOOM_OVERRIDE`: Optional acquisition zoom for both grid families. Defaults: WAC/static LTM 5 / polar 4; NAC LTM 11 / polar 10. Output retains native resolution (100 m for static-only).
- `MAX_OUTPUT_PIXELS`: Per-chip pixel-count safety limit before acquisition; reduce large NAC AOIs. Memory also scales with band count.
- `MAX_WORKERS`: Concurrent chip-creation workers.
- `INDEX_WORKER_COUNT`: Workers used to prepare source indexes.
- `SPLIT_CONFIG`: `NoSplitConfig` writes directly to `chips/` and `labels/`, without train/val/test directories.
- `OUTPUT_BASE_DIR`: Parent directory for timestamped runs under this notebook’s outputs.


In [ ]:
# Study setup
PROJECT_DATA_DIR = Path("/explore/nobackup/projects/lfm")
HEMISPHERE = "north"  # "north" or "south".
MODALITY = "wac"  # "wac", "nac", or "static".

# Imagery inputs
INCLUDE_STATIC = False  # Start with imagery only; verify static coverage separately.
SOURCE_RASTER = None  # Set an explicit Path for your source, especially NAC. Otherwise it will default to the static WAC polar .vrt file.
WAC_LAYOUT = "single"  # Polar mosaic is not the seven-band VIS/UV product.
DYNAMIC_GLOB = None  # Optional glob selecting this product's companion files.
POLAR_WAC_DIR = PROJECT_DATA_DIR / "processed_data/Lunar/Static_final/LROC/WAC/wac_glob_morf_mos"
STATIC_DATA_DIR = PROJECT_DATA_DIR / "staticLinks"

# Crater labels
LABEL_PATH = PROJECT_DATA_DIR / "example_notebook_inputs" / "WAC_GLOBAL_P900N0000_100M.eqc.iau2.LPS_N_label_craters.gpkg"  # Set your HPC copy's path if different.
LABEL_KIND = "auto"  # For instance TIFFs use "raster_instance".
LABEL_LAYER = "craters"

# Study area and additional AOIs
AOI_NWSE = (86.1, -48.7, 83.95, -19.5) if HEMISPHERE == "north" else (-85.9, -0.1, -86.1, 0.1)
RUN_SEAM_EXAMPLE = HEMISPHERE == "north" and MODALITY == "wac"  # Add the labeled north seam query.
SEAM_AOI_NWSE = (86.1, -48.7, 81.95, -19.5)  # Includes IDs 13–20 and crosses 82°N.
RUN_POLE_EXAMPLE = False  # Enable after confirming source and label coverage near the pole.
POLE_AOI_NWSE = (90, -180, 89.5, 180) if HEMISPHERE == 'north' else (-89.5, -180, -90, 180)
# Antimeridian alternative; confirm your imagery and labels cover this area:
# AOI_NWSE = (86.01, 179.98, 86.0, -179.98) if HEMISPHERE == 'north' else (-86.0, 179.98, -86.01, -179.98)

# Execution and outputs
SAMPLE_ID = f"{MODALITY}_{HEMISPHERE}_aoi_1"  # Unique per study area.
ZOOM_OVERRIDE = None
MAX_OUTPUT_PIXELS = 4_000_000  # Bounds per-band memory; static adds 63 bands.
MAX_WORKERS = 1
INDEX_WORKER_COUNT = 1
SPLIT_CONFIG = NoSplitConfig(seed=42)
OUTPUT_BASE_DIR = repo_root / "notebooks/outputs/chip_creation_polar"

### Validation coverage and controls

Run one hemisphere/source/label configuration at a time. Select extra real-data cases only after checking coverage; the presets are geometric examples, **not known annotated study areas**. Keep the output-size limit for NAC.

Repeat applicable geographic cases for **north and south**, WAC/NAC, dynamic-only / dynamic+static / static-only, and GPKG / semantic TIFF / instance TIFF. TIFFs require full realized-grid coverage; `LABEL_KIND="raster_instance"` is required for instance TIFFs. A single shared product stays in one split intentionally. Real NAC checks remain blocked until suitable imagery is available.

- `EXTRA_REAL_CASES`: selects presets below; empty preserves the current run.
- `CASE_LABEL_PATHS`: optional per-case finished labels; otherwise uses `LABEL_PATH`. No spatial identity matching is inferred.
- `RUN_PARALLEL_REPLAY`: reruns the actual batch twice (1 and `REPLAY_WORKERS` workers); potentially expensive.
- `RUN_REGRESSIONS`: runs the full chip suite plus tiling coverage/routing tests; may be run independently after imports and configuration. Skips are **incomplete validation**, not passes.


In [ ]:
EXTRA_REAL_CASES = ()  # e.g. ("antimeridian", "seam_antimeridian", "boundary_polar", "boundary_ltm")
CASE_LABEL_PATHS = {}  # e.g. {"antimeridian": Path("/path/to/finished_labels.gpkg")}
RUN_PARALLEL_REPLAY = False  # Two extra real-data runs; use at least two requests.
REPLAY_WORKERS = 2
RUN_REGRESSIONS = False  # Enable on the GDAL-enabled HPC kernel.

## 2. Resolve labels and inspect the output grid

No imagery acquisition occurs here. The realized rectangular raster may extend slightly beyond the input AOI due to reprojection and outward pixel rounding; labels must cover the entire realized grid. Review dimensions and label association before continuing. Canonical static coverage can be partial or absent; verified empty channels retain -32768 with warnings, while corrupt/missing inputs still fail.


In [ ]:
if HEMISPHERE not in ("north", "south") or MODALITY not in ("wac", "nac", "static"):
    raise ValueError("Choose a supported hemisphere and modality.")
if WAC_LAYOUT not in ("single", "vis_uv"):
    raise ValueError("WAC_LAYOUT must be 'single' or 'vis_uv'.")
requested_aoi = GeographicAOI(*AOI_NWSE)
family = chip_grid_family(requested_aoi)
expected_family = "lps_n" if HEMISPHERE == "north" else "lps_s"
if not (requested_aoi.upper_left_latitude > 82 if HEMISPHERE == 'north' else requested_aoi.lower_right_latitude < -82):
    raise ValueError("AOI must reach the selected polar hemisphere; crossing its 82° seam is allowed.")
static_only = MODALITY == "static"
source_grid = None
selectors = ()
GROUP_NAME = "polar"
if not static_only:
    if SOURCE_RASTER is None:
        if MODALITY != "wac" or WAC_LAYOUT != "single":
            raise ValueError("Set SOURCE_RASTER to your original imagery raster.")
        matches = sorted(POLAR_WAC_DIR.glob(f"*{expected_family.upper()}.vrt"))
        if len(matches) != 1:
            raise ValueError(f"Expected one polar VRT; found {len(matches)}. Set SOURCE_RASTER explicitly.")
        SOURCE_RASTER = matches[0]
    SOURCE_RASTER = Path(SOURCE_RASTER)
    source_grid = read_source_grid(
        SOURCE_RASTER, expected_band_count=5 if MODALITY == "wac" and WAC_LAYOUT == "vis_uv" else 1)
    PRODUCT_ID = lunar_product_id_from_raster_path(SOURCE_RASTER)
    selectors = (SourceSelector(GROUP_NAME, MODALITY, PRODUCT_ID),)
LABEL_PATH = latest_crater_label_path() if LABEL_PATH is None else Path(LABEL_PATH)
if not LABEL_PATH.is_file():
    raise FileNotFoundError(LABEL_PATH)
label_input = LabelInput(LABEL_PATH, kind=LABEL_KIND, relation="clip_to_target",
                         layer=LABEL_LAYER if LABEL_PATH.suffix.lower() == ".gpkg" else None)
request = chip_request_from_aoi(
    sample_id=SAMPLE_ID, geographic_aoi=requested_aoi,
    source_grid=source_grid, static_only=static_only,
    split_group_key=SAMPLE_ID if static_only else PRODUCT_ID,
    source_selectors=selectors, label_input=label_input,
)
if request.target_grid.width * request.target_grid.height > MAX_OUTPUT_PIXELS:
    raise ValueError("Output exceeds MAX_OUTPUT_PIXELS; reduce AOI or explicitly raise the limit.")
ZOOM_LEVEL = default_chip_zoom(request.geographic_aoi, MODALITY) if ZOOM_OVERRIDE is None else ZOOM_OVERRIDE
LTM_ZOOM = (11 if MODALITY == 'nac' else 5) if ZOOM_OVERRIDE is None else ZOOM_OVERRIDE
POLAR_ZOOM = (10 if MODALITY == 'nac' else 4) if ZOOM_OVERRIDE is None else ZOOM_OVERRIDE
OUTPUT_ROOT = OUTPUT_BASE_DIR / datetime.now().strftime("%Y%m%dT%H%M%S_%f")
print(f"Labels: {LABEL_PATH}")
print(f"Center/sole family: {family.value} | acquisition zooms: LTM={LTM_ZOOM}, polar={POLAR_ZOOM}")
print(f"Output: {request.target_grid.width} x {request.target_grid.height} | affine: {request.target_grid.transform}")
print(f"Destination: {OUTPUT_ROOT}")
requests = [request]


### Additional query: labeled north seam

This reuses the source grid, labels and product selector, with a distinct sample ID. The default AOI includes craters 13–20 from `(2).gpkg` and extends below 82°N (about 2.2 million pixels at 100 m). No supplied outlines cross the seam itself: inspect imagery continuity there and label alignment farther north. The output rectangle may extend beyond the geographic AOI. Both results share one manifest and are plotted below.


In [ ]:
requests = [request]  # Reset so rerunning this cell never duplicates the query.
if RUN_SEAM_EXAMPLE:
    seam_aoi = GeographicAOI(*SEAM_AOI_NWSE)
    boundary = 82 if HEMISPHERE == "north" else -82
    if not seam_aoi.lower_right_latitude < boundary < seam_aoi.upper_left_latitude:
        raise ValueError("SEAM_AOI_NWSE must cross the selected hemisphere's 82° seam.")
    seam_request = chip_request_from_aoi(
        sample_id=f"{SAMPLE_ID}_seam", geographic_aoi=seam_aoi,
        source_grid=source_grid, static_only=static_only,
        split_group_key=request.split_group_key,
        source_selectors=selectors, label_input=label_input,
    )
    grid = seam_request.target_grid
    if grid.width * grid.height > MAX_OUTPUT_PIXELS:
        raise ValueError("Seam output exceeds MAX_OUTPUT_PIXELS; reduce SEAM_AOI_NWSE.")
    requests.append(seam_request)
    print(f"Seam query: {SEAM_AOI_NWSE} | {grid.width} x {grid.height} pixels")


### Optional query: pole-containing rectangle

Enable `RUN_POLE_EXAMPLE` only with imagery and finished labels covering the pole. A full-longitude cap becomes an enclosing native-grid rectangle: corner imagery and labels outside the cap are retained, not masked. Acquisition covers that entire rectangle. The supplied example annotations are farther from the pole, so an all-background result here is not evidence of a correctly annotated negative sample. Large rectangles extending below the polar coverage boundary are rejected.


In [ ]:
requests = [item for item in requests if item.sample_id != f"{SAMPLE_ID}_pole"]
if RUN_POLE_EXAMPLE:
    pole_aoi = GeographicAOI(*POLE_AOI_NWSE)
    pole_latitude = pole_aoi.upper_left_latitude if HEMISPHERE == "north" else pole_aoi.lower_right_latitude
    if pole_latitude != (90 if HEMISPHERE == "north" else -90):
        raise ValueError("POLE_AOI_NWSE must reach the selected hemisphere's pole.")
    pole_request = chip_request_from_aoi(
        sample_id=f"{SAMPLE_ID}_pole", geographic_aoi=pole_aoi,
        source_grid=source_grid, static_only=static_only,
        split_group_key=request.split_group_key,
        source_selectors=selectors, label_input=label_input,
    )
    grid = pole_request.target_grid
    if grid.width * grid.height > MAX_OUTPUT_PIXELS:
        raise ValueError("Pole output exceeds MAX_OUTPUT_PIXELS; reduce the cap.")
    requests.append(pole_request)
    print(f"Pole query: {POLE_AOI_NWSE} | {grid.width} x {grid.height} pixels; corners retained")
    print("Acquisition envelope:", pole_request.geographic_aoi)


### Additional real-data geographic cases

These presets stay small for WAC; reduce them further for NAC. Exact-boundary inputs may acquire both families after the native rectangle is rounded outward—inspect the **realized** query parts, not just the input latitude. The combined seam/antimeridian case should not duplicate labels or introduce a blank stripe.

Presets use the selected source lattice and selectors. Unsupported source coverage is a meaningful failure, not an automatic reason to replace imagery or labels. Changing hemisphere does not change the label path for you.


In [ ]:
from lfm.data_processing.chip.chip_requests import geographic_query_parts

north_presets = {
    "antimeridian": (86.01, 179.98, 86.0, -179.98),
    "seam_antimeridian": (82.01, 179.98, 81.99, -179.98),
    "boundary_polar": (82.01, 10.0, 82.0, 10.02),
    "boundary_ltm": (82.0, 10.0, 81.99, 10.02),
}
presets = {name: bounds if HEMISPHERE == "north" else (-bounds[2], bounds[1], -bounds[0], bounds[3])
           for name, bounds in north_presets.items()}
unknown = set(EXTRA_REAL_CASES) - presets.keys()
if unknown:
    raise ValueError(f"Unknown extra cases: {sorted(unknown)}")
extra_ids = {f"{SAMPLE_ID}_{name}" for name in presets}
requests = [item for item in requests if item.sample_id not in extra_ids]  # Rerun-safe.
for name in dict.fromkeys(EXTRA_REAL_CASES):
    path = Path(CASE_LABEL_PATHS.get(name, LABEL_PATH))
    case_label = LabelInput(path, kind=LABEL_KIND, relation="clip_to_target",
                            layer=LABEL_LAYER if path.suffix.lower() == ".gpkg" else None)
    item = chip_request_from_aoi(
        sample_id=f"{SAMPLE_ID}_{name}", geographic_aoi=GeographicAOI(*presets[name]),
        source_grid=source_grid, static_only=static_only,
        split_group_key=request.split_group_key, source_selectors=selectors, label_input=case_label,
    )
    grid = item.target_grid
    if grid.width * grid.height > MAX_OUTPUT_PIXELS:
        raise ValueError(f"{name} exceeds MAX_OUTPUT_PIXELS; reduce its preset.")
    requests.append(item)
    print(name, "| labels:", path, "| pixels:", (grid.width, grid.height))
    print("Realized query parts:", geographic_query_parts(item.geographic_aoi))


## 3. Prepare enabled indexes and configure chips

Dynamic indexing is restricted to the chosen file/product and stored in this run's directory. Shared static indexes remain read-only; custom static collections use a per-clone cache. Disabled collections are not inspected. Single-band polar WAC is explicitly selected as one band, never forced into the seven-band contract.


In [ ]:
sources, modalities = [], []
if not static_only:
    pattern = DYNAMIC_GLOB or (
        f"{PRODUCT_ID}.prj.*.mos.tif" if MODALITY == "wac" and WAC_LAYOUT == "vis_uv"
        else SOURCE_RASTER.name)
    index_config = VectorIndexBuildConfig(
        data_dir=SOURCE_RASTER.parent, index_path=OUTPUT_ROOT / "indexes/dynamic.gpkg",
        image_glob=pattern, worker_count=INDEX_WORKER_COUNT)
    if SOURCE_RASTER.resolve() not in {p.resolve() for p in discover_raster_paths(index_config)}:
        raise ValueError('DYNAMIC_GLOB must include SOURCE_RASTER.')
    index = ensure_vector_index(index_config, stdout=sys.stdout).index_path
    sources.append(TileSourceConfig(
        name=MODALITY, data_dir=SOURCE_RASTER.parent, index_path=index,
        selection_mode="product_id", preserve_source_nodata=True, required=True))
    single = MODALITY == "nac" or WAC_LAYOUT == "single"
    modalities.append(OutputModalityConfig(
        GROUP_NAME, MODALITY, MODALITY,
        band_indices=(1,) if single else None,
        output_band_names=(MODALITY,) if single else None))
if static_only or INCLUDE_STATIC:
    resolution = resolve_notebook_source_index(
        source_name="static", data_dir=STATIC_DATA_DIR, cache_dir=OUTPUT_BASE_DIR / "indexes")
    index = ensure_vector_index(VectorIndexBuildConfig(
        data_dir=resolution.data_dir, index_path=resolution.index_path,
        rebuild_invalid_index=resolution.rebuild_invalid_index,
        worker_count=INDEX_WORKER_COUNT), stdout=sys.stdout).index_path
    sources.append(make_static_source(data_dir=STATIC_DATA_DIR, index_path=index))
    modalities.append(OutputModalityConfig(GROUP_NAME, "static", "static"))
chip_config = ChipConfig(
    output_root=OUTPUT_ROOT, label_source=LABEL_PATH,
    acquisition_groups=(AcquisitionGroupConfig(
        GROUP_NAME, TileConfig(OUTPUT_ROOT / ".tiling-template", ZOOM_LEVEL, tuple(sources)),
        ltm_zoom_level=LTM_ZOOM, polar_zoom_level=POLAR_ZOOM),),
    output_modalities=tuple(modalities), split_config=SPLIT_CONFIG,
)


## 4. Create and inspect

Per-sample errors are reported without silently treating failures as successful chips. Review the manifest, warnings and overlay. Run north/south, dynamic-only/mixed/static-only cases separately before relying on a new collection.


In [ ]:
batch = create_chips(requests, chip_config, max_workers=MAX_WORKERS)
print(f"Manifest: {batch.manifest_path}")
for result in batch.results:
    print(result.request.sample_id, result.status, result.message)
    for diagnostic in result.diagnostics:
        print(f"{diagnostic.stage}/{diagnostic.code}: {diagnostic.message}")
    if result.status == "success":
        plot_chip_result(result, display_band_keyword="elv" if static_only else MODALITY)
